In [ ]:
!git clone https://github.com/marcoshernanz/llm-lab.git /kaggle/working/llm-lab
%cd /kaggle/working/llm-lab
!git checkout milestone-023
!python -m pip install -q -U pip
!python -m pip install -q -U "jax[tpu]" flax optax numpy pyarrow datasets huggingface-hub

In [ ]:
"""Download the public tokenizer and shard files from Hugging Face."""

from pathlib import Path
from huggingface_hub import snapshot_download

HF_DATASET_REPO = "marcoshernanz/llm-lab-fineweb-edu-sample10bt-bpe-16384"
LOCAL_DATA_ROOT = Path("/kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384")
LOCAL_TOKENIZER_PATH = LOCAL_DATA_ROOT / "fineweb_edu_sample10bt_bpe_16384.json"

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=str(LOCAL_DATA_ROOT),
    allow_patterns=["*.json", "*.npy"],
    token=False,
)

print("local_dataset_root=", LOCAL_DATA_ROOT)
print("local_tokenizer_path=", LOCAL_TOKENIZER_PATH)
print("downloaded_files=")
for path in sorted(LOCAL_DATA_ROOT.iterdir()):
    print("  ", path.name)


In [ ]:
!PYTHONPATH=/kaggle/working/llm-lab python experiments/023_tpu_fineweb_edu_observability.py \
  --token-shard-root /kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384 \
  --tokenizer-path /kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384/fineweb_edu_sample10bt_bpe_16384.json \
  --artifacts-root /kaggle/working/artifacts/experiments \
  --execution-target "Kaggle TPU v5e-8" \
  --max-train-shards 10 \
  --train-steps 50000 \
  --train-chunk-length 100 \
  --batch-size 64 \
  --learning-rate 0.05

In [ ]:
"""Display the newest loss curve and metadata written by the milestone-023 run."""

from IPython.display import SVG, display
import json
from pathlib import Path

run_dirs = sorted(Path("/kaggle/working/artifacts/experiments/023_tpu_fineweb_edu_observability").glob("*"))
if not run_dirs:
    raise FileNotFoundError("No milestone-023 artifact directory was created.")

latest_run_dir = run_dirs[-1]
print("latest_run_dir=", latest_run_dir)
display(SVG(filename=str(latest_run_dir / "loss_curve.svg")))
print(json.dumps(json.loads((latest_run_dir / "run_metadata.json").read_text(encoding="utf-8")), indent=2))
print((latest_run_dir / "sample.txt").read_text(encoding="utf-8"))


In [ ]:
"""Zip the latest run into /kaggle/working so Kaggle exposes it for download."""

from pathlib import Path
import shutil

EXPERIMENT_NAME = "023_tpu_fineweb_edu_observability"
RUNS_ROOT = Path("/kaggle/working/artifacts/experiments") / EXPERIMENT_NAME
OUTPUT_ROOT = Path("/kaggle/working")

run_dirs = sorted(path for path in RUNS_ROOT.glob("*") if path.is_dir())
if not run_dirs:
    raise FileNotFoundError(f"No runs found under {RUNS_ROOT}")

latest_run_dir = run_dirs[-1]
archive_base = OUTPUT_ROOT / latest_run_dir.name
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(latest_run_dir.parent),
        base_dir=latest_run_dir.name,
    )
)

print("latest_run_dir=", latest_run_dir)
print("archive_path=", archive_path)
print("download_from_kaggle_output_panel=", archive_path.name)
